# Thực nghiệm chính: Phát hiện gian lận thẻ tín dụng

Notebook này điều phối toàn bộ thực nghiệm trên Kaggle. Giai đoạn hiện tại thực hiện EDA, tiền xử lý, xuất dữ liệu CSV theo từng fold và nạp lại dữ liệu để chuẩn bị cho bước huấn luyện mô hình.

Repository cần được clone vào `/kaggle/working`. Dữ liệu sẽ được tải tự động bằng API Kaggle chính thức và lưu trong `/kaggle/working/FAIR_2026_Experiment/data/Raw_data`.

## 1. Cài đặt thư viện tiền xử lý

In [ ]:
!git clone https://github.com/kohi-vip/FAIR_2026_Experiment.git

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "imblearn": "imbalanced-learn",
    "kagglehub": "kagglehub",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
    "catboost": "catboost",
    "pytorch_tabnet": "pytorch-tabnet",
    "tqdm": "tqdm",
}
missing_packages = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]
if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )
    print(f"[READY] Đã cài: {missing_packages}")
else:
    print("[READY] Các thư viện đã có sẵn; không cần cài lại.")

## 2. Xác định repository đã clone

Cell dưới đây không clone lại repository. Nó tìm `Notebook/EDA.py`, chuyển thư mục làm việc về project root và thêm project vào Python path.

In [ ]:
from pathlib import Path
import os
import sys

EXPECTED_REPOSITORY_NAME = "FAIR_2026_Experiment"
KAGGLE_WORKING_ROOT = Path("/kaggle/working")

search_roots = [Path.cwd().resolve()]
if KAGGLE_WORKING_ROOT.is_dir():
    search_roots.insert(0, KAGGLE_WORKING_ROOT / EXPECTED_REPOSITORY_NAME)

project_candidates = []
for search_root in search_roots:
    if (search_root / "Notebook" / "EDA.py").is_file():
        project_candidates.append(search_root)

if not project_candidates and KAGGLE_WORKING_ROOT.is_dir():
    project_candidates = [
        path.parent.parent
        for path in KAGGLE_WORKING_ROOT.rglob("Notebook/EDA.py")
    ]

project_candidates = list(dict.fromkeys(path.resolve() for path in project_candidates))
if len(project_candidates) != 1:
    raise FileNotFoundError(
        "Không xác định duy nhất repository đã clone. "
        f"Các đường dẫn tìm thấy: {project_candidates}"
    )

PROJECT_ROOT = project_candidates[0]
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"Kaggle runtime: {KAGGLE_WORKING_ROOT.is_dir()}")

## 3. Cấu hình lần chạy

Khi chạy trên nhiều máy độc lập, chỉ thay đổi cell cấu hình này. Mặc định chỉ tiền xử lý MLG-ULB fold 1 vì IEEE-CIS và Sparkov dùng Dense One-Hot kết hợp SMOTE, có thể cần rất nhiều RAM.

In [ ]:
# Các dataset hợp lệ: "MLG_ULB", "IEEE_CIS", "Sparkov".
# Trên mỗi máy, nên đặt EDA_DATASETS trùng với PREPROCESS_DATASETS.
EDA_DATASETS = ("MLG_ULB",)
PREPROCESS_DATASETS = ("MLG_ULB",)
FOLDS_TO_EXPORT = (1,)

SHOW_EDA_PLOTS = True
OVERWRITE_EXISTING_OUTPUTS = False
MAX_ESTIMATED_DENSE_GB = 8.0

# Fold sẽ được nạp vào RAM để bước huấn luyện model sử dụng.
TRAIN_DATASET = "MLG_ULB"
TRAIN_FOLD = 1
MLG_DUPLICATE_VARIANT = "without_duplicates"

# Các MODEL_NAME hợp lệ (12 mô hình truyền thống + TabNet):
# - Linear & Statistical: "Logistic_Regression", "LDA"
# - Single Decision Tree: "Decision_Tree"
# - Bagging: "Random_Forest", "Extra_Trees"
# - Boosting: "AdaBoost", "Gradient_Boosting", "XGBoost", "LightGBM", "CatBoost"
# - Distance & Probability: "Naive_Bayes", "KNN"
# - Deep Learning: "TabNet"
# Mặc định xác nhận pipeline bằng Logistic Regression.
MODEL_NAME = "Logistic_Regression"
DECISION_THRESHOLD = 0.5
OVERWRITE_MODEL_RESULTS = False
TABNET_MAX_EPOCHS = 100
TABNET_PATIENCE = 15
PREDICTION_BATCH_SIZE = 8192

if TRAIN_DATASET not in PREPROCESS_DATASETS:
    raise ValueError("TRAIN_DATASET phải nằm trong PREPROCESS_DATASETS.")
if TRAIN_FOLD not in FOLDS_TO_EXPORT:
    raise ValueError("TRAIN_FOLD phải nằm trong FOLDS_TO_EXPORT.")

## 4. Import pipeline, tải và kiểm tra dữ liệu Kaggle

In [ ]:
import importlib
from pathlib import Path
import sys

# Import cell vẫn tự hoạt động nếu được chạy riêng sau khi clone repository.
eda_module_candidates = []
if "PROJECT_ROOT" in globals():
    configured_module = Path(PROJECT_ROOT) / "Notebook" / "EDA.py"
    if configured_module.is_file():
        eda_module_candidates.append(configured_module)

kaggle_working_root = Path("/kaggle/working")
expected_module = (
    kaggle_working_root / "FAIR_2026_Experiment" / "Notebook" / "EDA.py"
)
if expected_module.is_file():
    eda_module_candidates.append(expected_module)
if not eda_module_candidates and kaggle_working_root.is_dir():
    eda_module_candidates.extend(kaggle_working_root.rglob("Notebook/EDA.py"))

eda_module_candidates = list(
    dict.fromkeys(path.resolve() for path in eda_module_candidates)
)
if len(eda_module_candidates) != 1:
    raise FileNotFoundError(
        "Không tìm thấy duy nhất Notebook/EDA.py trong repository đã clone. "
        f"Kết quả: {eda_module_candidates}"
    )

PROJECT_ROOT = eda_module_candidates[0].parent.parent
project_root_text = str(PROJECT_ROOT)
sys.path = [project_root_text, *[p for p in sys.path if p != project_root_text]]
importlib.invalidate_caches()
print(f"[READY] Python path đã nhận repository: {PROJECT_ROOT}")

from Notebook.EDA import (
    download_kaggle_data,
    find_data_file,
    load_processed_fold,
    run_eda,
    run_preprocessing,
)

DATASET_FILES = {
    "MLG_ULB": ("creditcard.csv",),
    "IEEE_CIS": ("train_transaction.csv", "train_identity.csv"),
    "Sparkov": ("fraudTrain.csv", "fraudTest.csv"),
}

datasets_needed = tuple(dict.fromkeys((*EDA_DATASETS, *PREPROCESS_DATASETS)))
resolved_input_files = download_kaggle_data(datasets=datasets_needed)
for dataset_name in datasets_needed:
    for filename in DATASET_FILES[dataset_name]:
        resolved_input_files[filename] = find_data_file(filename)
        print(f"[READY] {filename}: {resolved_input_files[filename]}")

## 5. Phân tích khám phá dữ liệu (EDA)

Giai đoạn này chỉ đọc và phân tích dữ liệu thô, không thay đổi dữ liệu và không huấn luyện mô hình.

In [ ]:
eda_reports = run_eda(
    datasets=EDA_DATASETS,
    show_plots=SHOW_EDA_PLOTS,
)

print(f"\n[HOÀN TẤT] Đã chạy EDA cho: {EDA_DATASETS}")

## 6. Tiền xử lý và xuất CSV

Scaler, imputer, One-Hot Encoder và SMOTE chỉ được fit trên training fold. Các CSV được ghi vào `/kaggle/working/FAIR_2026_Experiment/data/Processed_data` khi chạy trên Kaggle.

In [ ]:
artifacts = run_preprocessing(
    datasets=PREPROCESS_DATASETS,
    folds=FOLDS_TO_EXPORT,
    overwrite=OVERWRITE_EXISTING_OUTPUTS,
    max_estimated_dense_gb=MAX_ESTIMATED_DENSE_GB,
)

if not artifacts:
    raise RuntimeError("Pipeline không tạo ra artifact CSV nào.")

print(f"\n[HOÀN TẤT] Đã tạo {len(artifacts)} bộ train/validation CSV.")
for artifact in artifacts:
    variant_label = f" | variant={artifact.variant}" if artifact.variant else ""
    print(f"- {artifact.dataset} | fold={artifact.fold}{variant_label}")
    print(f"  train: {artifact.train_csv}")
    print(f"  validation: {artifact.validation_csv}")

## 7. Nạp dữ liệu đã xử lý cho bước huấn luyện

Bốn biến `X_train`, `y_train`, `X_valid`, `y_valid` là đầu vào trực tiếp cho các model ở giai đoạn tiếp theo. Validation fold không được áp dụng SMOTE.

In [ ]:
selected_variant = (
    MLG_DUPLICATE_VARIANT if TRAIN_DATASET == "MLG_ULB" else None
)
X_train, y_train, X_valid, y_valid = load_processed_fold(
    TRAIN_DATASET,
    fold=TRAIN_FOLD,
    variant=selected_variant,
)

print(f"Dataset: {TRAIN_DATASET} | fold: {TRAIN_FOLD}")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_valid: {X_valid.shape} | y_valid: {y_valid.shape}")
print("\nPhân bố nhãn training sau SMOTE:")
print(y_train.value_counts().sort_index())
print("\nPhân bố nhãn validation gốc:")
print(y_valid.value_counts().sort_index())

## 8. Huấn luyện và đánh giá mô hình

Mặc định notebook huấn luyện `Logistic_Regression` từ factory đã được kiểm tra trong `12_model_test.ipynb`. Có thể đổi `MODEL_NAME` trong cell cấu hình sang một trong 12 model truyền thống hoặc `TabNet`. Không dùng validation fold để fit scaler, encoder, SMOTE hoặc model.

In [ ]:
from models.fraud_models import get_tabnet_model, get_traditional_models

RANDOM_STATE = 42
traditional_models = get_traditional_models(random_state=RANDOM_STATE)
available_model_names = (*traditional_models.keys(), "TabNet")
assert len(available_model_names) == 13
if MODEL_NAME not in available_model_names:
    raise ValueError(
        f"MODEL_NAME={MODEL_NAME!r} không hợp lệ. Chọn một trong {available_model_names}."
    )

selected_model = (
    get_tabnet_model(random_state=RANDOM_STATE)
    if MODEL_NAME == "TabNet"
    else traditional_models[MODEL_NAME]
)
model_class_path = f"{type(selected_model).__module__}.{type(selected_model).__name__}"

variant_directory = selected_variant or "default"
RESULTS_DIRECTORY = (
    PROJECT_ROOT / "results" / TRAIN_DATASET / variant_directory / f"fold_{TRAIN_FOLD:02d}"
)
RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RESULTS_DIRECTORY / f"metrics_{MODEL_NAME}.csv"
PREDICTIONS_PATH = RESULTS_DIRECTORY / f"predictions_{MODEL_NAME}.csv"
EVALUATION_FIGURE_PATH = RESULTS_DIRECTORY / f"evaluation_{MODEL_NAME}.png"
existing_result_paths = [
    path
    for path in (METRICS_PATH, PREDICTIONS_PATH, EVALUATION_FIGURE_PATH)
    if path.exists()
]
if existing_result_paths and not OVERWRITE_MODEL_RESULTS:
    raise FileExistsError(
        f"Kết quả đã tồn tại: {existing_result_paths}. "
        "Đặt OVERWRITE_MODEL_RESULTS=True nếu chủ động chạy lại."
    )

print(f"[READY] Model: {MODEL_NAME}")
print(f"Class: {model_class_path}")
print(f"Nơi lưu kết quả: {RESULTS_DIRECTORY}")

In [ ]:
from time import perf_counter
import numpy as np
from tqdm.auto import tqdm

training_started_at = perf_counter()
with tqdm(total=1, desc=f"Train {MODEL_NAME}", unit="model") as train_progress:
    if MODEL_NAME == "TabNet":
        X_train_for_model = X_train.to_numpy(dtype=np.float32)
        X_valid_for_model = X_valid.to_numpy(dtype=np.float32)
        y_train_for_model = y_train.to_numpy(dtype=np.int64)
        y_valid_for_model = y_valid.to_numpy(dtype=np.int64)
        selected_model.fit(
            X_train_for_model,
            y_train_for_model,
            eval_set=[(X_valid_for_model, y_valid_for_model)],
            eval_name=["validation"],
            eval_metric=["auc"],
            max_epochs=TABNET_MAX_EPOCHS,
            patience=TABNET_PATIENCE,
            batch_size=8192,
            virtual_batch_size=512,
            num_workers=2,
            drop_last=False,
        )
    else:
        X_train_for_model = X_train
        X_valid_for_model = X_valid
        y_train_for_model = y_train
        y_valid_for_model = y_valid
        selected_model.fit(X_train_for_model, y_train_for_model)
    train_progress.update(1)

training_seconds = perf_counter() - training_started_at
probability_batches = []
testing_started_at = perf_counter()
with tqdm(
    total=len(y_valid_for_model),
    desc=f"Test {MODEL_NAME}",
    unit="mẫu",
) as test_progress:
    for batch_start in range(0, len(y_valid_for_model), PREDICTION_BATCH_SIZE):
        batch_end = min(batch_start + PREDICTION_BATCH_SIZE, len(y_valid_for_model))
        if hasattr(X_valid_for_model, "iloc"):
            X_batch = X_valid_for_model.iloc[batch_start:batch_end]
        else:
            X_batch = X_valid_for_model[batch_start:batch_end]
        probability_batches.append(selected_model.predict_proba(X_batch)[:, 1])
        test_progress.update(batch_end - batch_start)
fraud_scores = np.concatenate(probability_batches)
testing_seconds = perf_counter() - testing_started_at
fraud_predictions = (fraud_scores >= DECISION_THRESHOLD).astype(np.int8)
print(f"[HOÀN TẤT] Huấn luyện {MODEL_NAME} trong {training_seconds:,.2f} giây.")
print(f"[HOÀN TẤT] Test/validation trong {testing_seconds:,.2f} giây.")

## 9. Ma trận nhầm lẫn và các chỉ số đánh giá

Quy ước lớp dương là giao dịch gian lận (`Class = 1`). Ma trận nhầm lẫn dùng thứ tự `[[TN, FP], [FN, TP]]`.

- **Precision** = `TP / (TP + FP)`.
- **Recall** = `TP / (TP + FN)`.
- **F1-Score** = `2 × Precision × Recall / (Precision + Recall)`.
- **ROC-AUC** được tính từ xác suất fraud trên toàn bộ các ngưỡng.
- **PR-AUC** là diện tích hình thang dưới đường cong Precision-Recall; Average Precision cũng được lưu riêng để tránh nhập nhằng giữa hai quy ước thường dùng.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    auc,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

matrix = confusion_matrix(y_valid_for_model, fraud_predictions, labels=[0, 1])
tn, fp, fn, tp = (int(value) for value in matrix.ravel())
precision = precision_score(
    y_valid_for_model, fraud_predictions, pos_label=1, zero_division=0
)
recall = recall_score(
    y_valid_for_model, fraud_predictions, pos_label=1, zero_division=0
)
f1 = f1_score(y_valid_for_model, fraud_predictions, pos_label=1, zero_division=0)
roc_auc = roc_auc_score(y_valid_for_model, fraud_scores)
curve_precision, curve_recall, _ = precision_recall_curve(
    y_valid_for_model, fraud_scores, pos_label=1
)
pr_auc = auc(curve_recall, curve_precision)
average_precision = average_precision_score(y_valid_for_model, fraud_scores)

manual_precision = tp / (tp + fp) if tp + fp else 0.0
manual_recall = tp / (tp + fn) if tp + fn else 0.0
manual_f1 = (
    2 * manual_precision * manual_recall / (manual_precision + manual_recall)
    if manual_precision + manual_recall
    else 0.0
)
assert np.isclose([precision, recall, f1], [manual_precision, manual_recall, manual_f1]).all()

metrics_result = pd.DataFrame(
    [{
        "dataset": TRAIN_DATASET,
        "duplicate_variant": selected_variant,
        "fold": TRAIN_FOLD,
        "model": MODEL_NAME,
        "model_class": model_class_path,
        "decision_threshold": DECISION_THRESHOLD,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "Average_Precision": average_precision,
        "training_seconds": training_seconds,
        "testing_seconds": testing_seconds,
        "train_rows": len(y_train_for_model),
        "validation_rows": len(y_valid_for_model),
    }]
)
display(metrics_result.T.rename(columns={0: "Kết quả"}))

figure, axes = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=["Hợp lệ (0)", "Gian lận (1)"],
).plot(ax=axes[0], colorbar=False, values_format="d")
axes[0].set_title(f"Confusion Matrix - {MODEL_NAME}")
RocCurveDisplay.from_predictions(
    y_valid_for_model, fraud_scores, pos_label=1, ax=axes[1], name=MODEL_NAME
)
axes[1].set_title(f"ROC Curve - AUC={roc_auc:.6f}")
PrecisionRecallDisplay.from_predictions(
    y_valid_for_model, fraud_scores, pos_label=1, ax=axes[2], name=MODEL_NAME
)
axes[2].set_title(f"Precision-Recall Curve - AUC={pr_auc:.6f}")
plt.tight_layout()
figure.savefig(EVALUATION_FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()

predictions_result = pd.DataFrame({
    "validation_row": np.arange(len(y_valid_for_model)),
    "y_true": np.asarray(y_valid_for_model, dtype=np.int8),
    "fraud_score": fraud_scores,
    "y_pred": fraud_predictions,
})
metrics_result.to_csv(METRICS_PATH, index=False)
predictions_result.to_csv(PREDICTIONS_PATH, index=False)
print(f"[READY] Metrics CSV: {METRICS_PATH}")
print(f"[READY] Predictions CSV: {PREDICTIONS_PATH}")
print(f"[READY] Evaluation figure: {EVALUATION_FIGURE_PATH}")